# 05 GSE65858_external_validation

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt

In [ ]:
df_min = pd.read_csv("HNSC_hypoxia_signature_lambda_min.csv")
df_1se = pd.read_csv("HNSC_hypoxia_signature_lambda_1se.csv")
needed_genes = sorted(set(df_min["Gene"]) | set(df_1se["Gene"]))
print("lambda.min (PRIMARY):", df_min.shape[0], "genes")
print("lambda.1se (SENSITIVITY):", df_1se.shape[0], "genes")
print("Union of genes needed from GSE65858:", len(needed_genes))

In [ ]:
probe_to_symbol = {}
with open("GPL10558_HumanHT-12_V4_0_R2_15002873_B.txt", "r", encoding="latin-1") as f:
    for line in f:
        if line.startswith("Species\tSource\tSearch_Key"):
            header = line.strip().split("\t")
            break
    probe_idx = header.index("Probe_Id")
    symbol_idx = header.index("Symbol")
    for line in f:
        if line.startswith("[Controls]") or line.strip() == "":
            break
        fields = line.rstrip("\n").split("\t")
        if len(fields) <= max(probe_idx, symbol_idx):
            continue
        pid, sym = fields[probe_idx].strip(), fields[symbol_idx].strip()
        if pid and sym:
            probe_to_symbol[pid] = sym

symbol_to_probe_ids = defaultdict(list)
for pid, sym in probe_to_symbol.items():
    if sym in needed_genes:
        symbol_to_probe_ids[sym].append(pid)

needed_probe_ids = set(p for probes in symbol_to_probe_ids.values() for p in probes)
print("Probes to extract:", len(needed_probe_ids))

In [ ]:
wb = openpyxl.load_workbook("GSE65858.xlsx", data_only=True, read_only=True)
ws = wb["Sheet1"]

header_row = list(ws.iter_rows(min_row=105, max_row=105, values_only=True))[0]
sample_ids = [str(v) for v in header_row[1:] if v]
print("Samples in GSE65858:", len(sample_ids))

expression_rows = {}
for row in ws.iter_rows(min_row=106, max_row=ws.max_row, values_only=True):
    probe_id = row[0]
    if probe_id in needed_probe_ids:
        expression_rows[probe_id] = row[1:1+len(sample_ids)]

gene_expr = {}
for gene, probes in symbol_to_probe_ids.items():
    vals = [expression_rows[p] for p in probes if p in expression_rows]
    if vals:
        gene_expr[gene] = np.array(vals, dtype=float).mean(axis=0)

expr_df = pd.DataFrame(gene_expr, index=sample_ids)
print("Gene-level expression matrix:", expr_df.shape, "| missing values:", expr_df.isna().sum().sum())

In [ ]:
def get_row_values(row_idx, n_samples):
    row = list(ws.iter_rows(min_row=row_idx, max_row=row_idx, max_col=1+n_samples, values_only=True))[0]
    vals = []
    for v in row[1:]:
        if v is None:
            vals.append(None); continue
        v = str(v)
        vals.append(v.split(":", 1)[1].strip() if ":" in v else v)
    return vals

n = len(sample_ids)
os_time = get_row_values(51, n)
os_event = get_row_values(52, n)

clin = pd.DataFrame({"Sample": sample_ids, "OS_time_days": os_time, "OS_event_raw": os_event})
clin["OS_time_days"] = pd.to_numeric(clin["OS_time_days"], errors="coerce")
clin["OS_months"] = clin["OS_time_days"] / 30.44
clin["Event"] = clin["OS_event_raw"].map({"TRUE": 1, "FALSE": 0})
clin = clin.dropna(subset=["OS_months", "Event"])
clin = clin[clin["OS_months"] > 0].set_index("Sample")

print("Patients with valid survival data:", clin.shape[0], "| Events:", int(clin["Event"].sum()))

In [ ]:
data = expr_df.join(clin[["OS_months", "Event"]], how="inner")
print("Patients with both expression and survival data:", data.shape[0], "| Events:", int(data["Event"].sum()))

def compute_locked_score(data, coef_df):
    score = pd.Series(0.0, index=data.index)
    for _, row in coef_df.iterrows():
        gene, coef = row["Gene"], row["Coefficient"]
        z = (data[gene] - data[gene].mean()) / data[gene].std()
        score = score + coef * z
    return score - score.mean()

data["Score_min"] = compute_locked_score(data, df_min)
data["Score_1se"] = compute_locked_score(data, df_1se)

In [ ]:
def run_validation(data, score_col, label):
    print("="*70)
    print(f"EXTERNAL VALIDATION: {label}")
    print("="*70)
    d = data[["OS_months", "Event", score_col]].dropna()
    cph = CoxPHFitter()
    cph.fit(d, duration_col="OS_months", event_col="Event")
    cph.print_summary()

    summ = cph.summary.loc[score_col]
    hr, lo, hi, pval = np.exp(summ["coef"]), np.exp(summ["coef lower 95%"]), np.exp(summ["coef upper 95%"]), summ["p"]
    cidx = cph.concordance_index_
    print(f"\n--> HR={hr:.3f} (95% CI {lo:.3f}-{hi:.3f}), p={pval:.4g}, C-index={cidx:.3f}")

    med = d[score_col].median()
    high, low = d[d[score_col] >= med], d[d[score_col] < med]
    lr = logrank_test(high["OS_months"], low["OS_months"], high["Event"], low["Event"])
    print(f"Log-rank p-value: {lr.p_value:.4g}")

    plt.figure(figsize=(7, 5.5))
    KaplanMeierFitter().fit(high["OS_months"], high["Event"], label=f"High (n={len(high)})").plot_survival_function(ci_show=True)
    KaplanMeierFitter().fit(low["OS_months"], low["Event"], label=f"Low (n={len(low)})").plot_survival_function(ci_show=True)
    plt.xlabel("Overall Survival (Months)"); plt.ylabel("Survival Probability")
    plt.title(f"GSE65858 External Validation: {label}")
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

    return dict(N=d.shape[0], events=int(d["Event"].sum()), HR=hr, CI_low=lo, CI_high=hi, p=pval, C_index=cidx, logrank_p=lr.p_value)

In [ ]:
result_min = run_validation(data, "Score_min", "lambda.min (PRIMARY, 32 genes)")

In [ ]:
result_1se = run_validation(data, "Score_1se", "lambda.1se (SENSITIVITY, 13 genes)")

In [ ]:
summary_df = pd.DataFrame([
    {"Model": "lambda.min (PRIMARY, 32 genes)", **result_min},
    {"Model": "lambda.1se (SENSITIVITY, 13 genes)", **result_1se},
])
summary_df.to_csv("GSE65858_validation_summary.csv", index=False)
summary_df